# Animação 3D do CSV gerado por t11.c

Este notebook usa o formato `frame,x,y,z,magnitude` produzido por [t11/t11.c](t11/t11.c) como referência para carregar, validar, analisar e animar os dados com `matplotlib` em 3D.

O fluxo abaixo recompila o programa, reconstrói o CSV, confere o esquema e gera uma animação temporal da nuvem de pontos por frame.

## 1. Compilar e Executar `t11.c` para Gerar o CSV

A célula seguinte recompila o programa em C e executa a simulação para garantir que o arquivo `animacao_completa.csv` esteja atualizado antes da análise.

In [ ]:
from pathlib import Path
import subprocess


def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "t11" / "t11.c").exists() and (candidate / "animacao_completa.csv").exists():
            return candidate
        if (candidate / "t11" / "t11.c").exists():
            return candidate
    raise FileNotFoundError("Não foi possível localizar o diretório raiz do projeto.")


project_root = find_project_root()
source_file = project_root / "t11" / "t11.c"
binary_file = project_root / "t11" / "t11.out"
csv_file = project_root / "animacao_completa.csv"

compile_cmd = ["gcc", str(source_file), "-lm", "-o", str(binary_file)]
subprocess.run(compile_cmd, cwd=project_root, check=True)
subprocess.run([str(binary_file)], cwd=project_root, check=True)

print(f"Raiz do projeto: {project_root}")
print(f"CSV gerado: {csv_file} -> {csv_file.exists()}")
print(f"Tamanho do CSV: {csv_file.stat().st_size if csv_file.exists() else 0} bytes")

## 2. Importar Bibliotecas e Definir Caminho do Arquivo

Aqui carregamos as bibliotecas principais e centralizamos os caminhos usados ao longo do notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from IPython.display import HTML, display
import plotly.express as px

plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print(f"CSV alvo: {csv_file}")
print(f"Arquivo existe: {csv_file.exists()}")

## 3. Carregar `animacao_completa.csv` com Pandas

A leitura abaixo confirma se o arquivo foi gerado corretamente por `t11.c` e mostra as primeiras linhas da tabela.

In [ ]:
df = pd.read_csv(csv_file)

print("Primeiras linhas:")
display(df.head())
print("\nTipos de dados:")
print(df.dtypes)
print(f"\nTotal de linhas: {len(df):,}")
print(f"Frames únicos: {df['frame'].nunique()}")

## 4. Validar Esquema do CSV (Referência do `t11`)

A estrutura esperada é `frame,x,y,z,magnitude`, com valores numéricos e sem campos vazios.

In [ ]:
expected_columns = ["frame", "x", "y", "z", "magnitude"]
column_order_ok = list(df.columns) == expected_columns
missing_columns = [column for column in expected_columns if column not in df.columns]
invalid_counts = df[expected_columns].isna().sum()

for column in expected_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

numeric_invalid = df[expected_columns].isna().sum()
frame_range = (int(df["frame"].min()), int(df["frame"].max()))
coord_ranges = {
    axis: (int(df[axis].min()), int(df[axis].max())) for axis in ["x", "y", "z"]
}
magnitude_range = (float(df["magnitude"].min()), float(df["magnitude"].max()))

print(f"Ordem das colunas correta: {column_order_ok}")
print(f"Colunas ausentes: {missing_columns}")
print("Valores nulos antes/depois da conversão:")
display(pd.DataFrame({"antes": invalid_counts, "depois": numeric_invalid}))
print(f"Intervalo de frames: {frame_range}")
print(f"Intervalos de coordenadas: {coord_ranges}")
print(f"Intervalo de magnitude: {magnitude_range}")

schema_ok = column_order_ok and not missing_columns and numeric_invalid.sum() == 0
print(f"Esquema válido: {schema_ok}")

## 5. Calcular Estatísticas por `frame`

Agrupamos os dados por frame para acompanhar a intensidade média, o pico e a quantidade de pontos ativos ao longo do tempo.

In [ ]:
frame_stats = (
    df.groupby("frame")
    .agg(
        pontos_ativos=("magnitude", "size"),
        magnitude_media=("magnitude", "mean"),
        magnitude_max=("magnitude", "max"),
        magnitude_p90=("magnitude", lambda s: s.quantile(0.90)),
        magnitude_p99=("magnitude", lambda s: s.quantile(0.99)),
    )
    .reset_index()
)

display(frame_stats.head())
print(f"Frames analisados: {len(frame_stats)}")

## 6. Visualizar Evolução Temporal da Magnitude

Os gráficos abaixo ajudam a observar a dissipação da perturbação ao longo dos frames e a evolução do número de pontos acima do limiar.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

axes[0].plot(frame_stats["frame"], frame_stats["magnitude_media"], marker="o", label="Média")
axes[0].plot(frame_stats["frame"], frame_stats["magnitude_max"], marker="s", label="Máximo")
axes[0].plot(frame_stats["frame"], frame_stats["magnitude_p90"], marker="^", label="P90")
axes[0].set_ylabel("Magnitude")
axes[0].set_title("Evolução da magnitude por frame")
axes[0].legend()

axes[1].plot(frame_stats["frame"], frame_stats["pontos_ativos"], marker="o", color="tab:orange")
axes[1].set_xlabel("Frame")
axes[1].set_ylabel("Pontos ativos")
axes[1].set_title("Quantidade de pontos acima do limiar")

plt.tight_layout()
plt.show()

## 7. Renderizar Nuvem 3D por Frame

A célula a seguir prepara uma função reutilizável para desenhar a nuvem de pontos de um frame específico em 3D, com cor por magnitude e limites fixos.

In [ ]:
frame_min = int(df["frame"].min())
frame_max = int(df["frame"].max())
coord_min = int(df[["x", "y", "z"]].min().min())
coord_max = int(df[["x", "y", "z"]].max().max())


def plot_frame_3d(frame_id: int, magnitude_min: float = 0.0, max_points: int | None = None):
    frame_df = df[(df["frame"] == frame_id) & (df["magnitude"] >= magnitude_min)].copy()
    if max_points is not None and len(frame_df) > max_points:
        frame_df = frame_df.sample(max_points, random_state=42)

    fig = plt.figure(figsize=(9, 8))
    ax = fig.add_subplot(111, projection="3d")
    scatter = ax.scatter(
        frame_df["x"],
        frame_df["y"],
        frame_df["z"],
        c=frame_df["magnitude"],
        cmap="viridis",
        s=10,
        alpha=0.85,
    )
    ax.set_xlim(coord_min, coord_max)
    ax.set_ylim(coord_min, coord_max)
    ax.set_zlim(coord_min, coord_max)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(f"Frame {frame_id} - pontos: {len(frame_df)}")
    fig.colorbar(scatter, ax=ax, shrink=0.7, label="Magnitude")
    plt.show()


plot_frame_3d(frame_min)

## 8. Criar Animação dos Frames

A animação usa `frame` como eixo temporal e mantém a escala espacial fixa para que a dissipação seja comparável entre quadros.

In [ ]:
selected_frames = sorted(frame_stats["frame"].tolist())
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

scatter = ax.scatter([], [], [], c=[], cmap="viridis", s=8, alpha=0.85)
ax.set_xlim(coord_min, coord_max)
ax.set_ylim(coord_min, coord_max)
ax.set_zlim(coord_min, coord_max)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
colorbar = fig.colorbar(scatter, ax=ax, shrink=0.7, label="Magnitude")


def update(frame_id):
    frame_df = df[df["frame"] == frame_id]
    if len(frame_df) == 0:
        return scatter,

    scatter._offsets3d = (
        frame_df["x"].to_numpy(),
        frame_df["y"].to_numpy(),
        frame_df["z"].to_numpy(),
    )
    scatter.set_array(frame_df["magnitude"].to_numpy())
    scatter.set_clim(vmin=float(df["magnitude"].min()), vmax=float(df["magnitude"].max()))
    ax.set_title(f"Frame {frame_id} | pontos: {len(frame_df)}")
    return scatter,

anim = FuncAnimation(fig, update, frames=selected_frames, interval=250, blit=False)
HTML(anim.to_jshtml())

## 9. Salvar Artefatos (CSV Limpo e Vídeo/GIF)

Este bloco salva uma versão ordenada do CSV e tenta exportar a animação em `GIF` ou `MP4`, dependendo das bibliotecas disponíveis no ambiente.

In [ ]:
output_csv = project_root / "animacao_completa_limpo.csv"
df.sort_values(["frame", "x", "y", "z"]).to_csv(output_csv, index=False)
print(f"CSV limpo salvo em: {output_csv}")

output_dir = project_root / "t11"
output_gif = output_dir / "animacao_3d.gif"
output_mp4 = output_dir / "animacao_3d.mp4"

try:
    anim.save(output_gif, writer="pillow", fps=4)
    print(f"Animação salva em GIF: {output_gif}")
except Exception as exc:
    print(f"Falha ao salvar GIF: {exc}")
    try:
        anim.save(output_mp4, writer="ffmpeg", fps=4)
        print(f"Animação salva em MP4: {output_mp4}")
    except Exception as exc2:
        print(f"Falha ao salvar MP4: {exc2}")
        print("A animação continua disponível inline no notebook.")